# Phase 1 results: every version, every rule, every dataset

**What varies.** How the graph is re-embedded when a new entity arrives (the *version*), and when the OCSVM is
fitted again (the *rule*). Everything else is the same in every run (64 numbers per vector, window 250,
OCSVM ν = 0.1, γ 'scale', α = 0.025, keep 10% after an alarm, seed 42).

| Version | Re-embedding |
|---|---|
| scratch, 2 / 10 / 20 epochs | a new model on the grown graph, the same number of epochs for every model (each model's chosen learning rate) |
| scratch, MRR-chosen epochs (v1) | a new model on the grown graph, epochs chosen per model by validation MRR (20-100) |
| warm start (v3) | the previous vectors, a random vector for the new entity, a few epochs over the whole graph (chosen by `warm_check.py`) |

| Rule | The OCSVM is fitted again (on the same reference triples, with the current vectors) |
|---|---|
| R1 sphere k=1 | at a new entity, if the new triple is outside Antoine's sphere (ratio > 1) |
| R2 always | at every new entity |
| R3 sphere ~10/50/90% | Antoine's sphere with a smaller limit, set so it would fire at about 10/50/90% of new entities |
| R4 every 250/50/10 | after every X stream triples |

After an alarm every rule does Antoine's procedure: keep 25, refill 225 untested, re-embed, new reference.

**Data.** CASE1 (Gene-Disease, graph up to 2002, change 2005: cancer to infectious), CASE2 (miRNA-Disease,
graph up to 2018, change 2022: infectious to cancer), full data (Gene-Disease, graph up to 2010, no known change).
The change is placed at the first stream triple of the change year (CASE1: stream triple 567), whether or
not a run tested it.

**How to read the tables.** One run = one model with one composition. Where 10 runs are summarised (5 models
x 2 compositions), the table gives counts of runs, or the median with the lowest and highest value.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 400)
HERE = Path.cwd() if (Path.cwd() / 'results_helpers.py').exists() else Path.cwd() / 'notebooks'
sys.path.insert(0, str(HERE))
import results_helpers as rh

t = rh.load()
print(f'{len(t)} finished runs')

## 1. What has been run (finished / planned)

In [ ]:
rh.run_overview(t)

## 2. Correctness checks (details in `checks.ipynb`)

In [ ]:
c = rh.read_csv(rh.paths()['checks'])
print(f"{len(c)} checks, {int(c['passed'].sum())} passed" if len(c) else 'checks not run yet')
c[~c['passed']] if len(c) else None

## 3. Embedding quality

Test MRR (filtered link prediction on 2% of the starting graph held out from training) per model and version.
Random guessing is about 0.0001-0.0002. The warm start starts from the v1 embedding.

In [ ]:
rh.quality_table()

### Warm start: how many epochs per update

`warm_check.py` adds one window of stream triples (250) to the graph, continues training from the old vectors
for 1, 2, 3, 5, 10 epochs at the chosen learning rate and one fifth of it (Wewer et al. 2021), and measures
(a) the old triples' validation MRR, (b) how the old OCSVM judges the reference triples under the updated
vectors, (c) the MRR of the added triples. The rule fixed before the measurements: pass (a) >= 95% of before
and (b) a rise of at most 0.05, then the best (c), then the fewest epochs. `passes = False` means no setting
passed and the gentlest one was taken.

In [ ]:
for dataset, table in rh.warm_tables().items():
    print(dataset); display(table)

## 4. CASE1: the change is in 2005 (cancer to infectious disease)

### Summary over the 10 runs of each version and rule

In [ ]:
rh.case_summary(t, 'case1')

### Every run: alarms before the change, alarms after it, delay

In [ ]:
rh.heatmap(t, 'case1', 'alarms_pre', 'alarms before the change')
rh.heatmap(t, 'case1', 'alarms_post', 'alarms after the change', cmap='Blues')
rh.heatmap(t, 'case1', 'detection_delay', 'delay of the first alarm after the change (stream triples)', cmap='Purples')

### Where the alarms fall relative to the change

In [ ]:
rh.alarm_offsets_grid('case1')

### Outlier rate over the stream (TransE, Hadamard; the other runs are in `analysis/figures/`)

Blue line = window outlier rate at tested triples, grey = reference rate, red = alarms, orange = peaks (numbered by
height, with their date), light blue = untested refill triples as the OCSVM in use at the alarm sees them,
ticks at the bottom = re-embeds (green = OCSVM retrained), dashed = the change.

In [ ]:
rh.show_run_figures('case1', rules=('sphere1', 'always', 'every50'))

### Peaks: highest and second-highest, and their distance to the change

In [ ]:
rh.peak_distance_table(t, 'case1')

### Peak dates over all runs (for finding the papers: `analysis/literature/`)

In [ ]:
rh.peak_dates('case1')

## 5. CASE2: the change is in 2022 (infectious disease to cancer)

### Summary over the 10 runs of each version and rule

In [ ]:
rh.case_summary(t, 'case2')

### Every run: alarms before the change, alarms after it, delay

In [ ]:
rh.heatmap(t, 'case2', 'alarms_pre', 'alarms before the change')
rh.heatmap(t, 'case2', 'alarms_post', 'alarms after the change', cmap='Blues')
rh.heatmap(t, 'case2', 'detection_delay', 'delay of the first alarm after the change (stream triples)', cmap='Purples')

### Where the alarms fall relative to the change

In [ ]:
rh.alarm_offsets_grid('case2')

### Outlier rate over the stream (TransE, Hadamard; the other runs are in `analysis/figures/`)

Blue line = window outlier rate at tested triples, grey = reference rate, red = alarms, orange = peaks (numbered by
height, with their date), light blue = untested refill triples as the OCSVM in use at the alarm sees them,
ticks at the bottom = re-embeds (green = OCSVM retrained), dashed = the change.

In [ ]:
rh.show_run_figures('case2', rules=('sphere1', 'always', 'every50'))

### Peaks: highest and second-highest, and their distance to the change

In [ ]:
rh.peak_distance_table(t, 'case2')

### Peak dates over all runs (for finding the papers: `analysis/literature/`)

In [ ]:
rh.peak_dates('case2')

## 6. Full data (no known change): where are the alarms and the peaks?

The v1 runs with Rules 1 and 2 are the earlier ones (5 to 27 hours each); v1 is not run with Rules 3 and 4
on the full data. The 10-epoch runs of TransE, TransH, DistMult and RotatE with Rules 1 and 2 are the earlier
10-epoch runs (same settings, `imported_from` in their summary).

In [ ]:
rh.full_alarms_by_year(t)

In [ ]:
rh.heatmap(t, 'full', 'num_detected_drifts', 'number of alarms')

In [ ]:
rh.show_run_figures('full', rules=('sphere1', 'always'))

### Peak dates on the full data, ordered by how many runs have a peak there

In [ ]:
rh.peak_dates('full', top=50)

## 7. Sensitivity of the detector settings (one setting changed at a time)

Version scratch 20 epochs, Rule 2, both cases, 10 runs per setting. `base` = the Phase 1 values.
`nu`: OCSVM ν; `win`: window size; `alpha`: test level; `keep`: share of the window kept after an alarm;
`seed`: random seed of the embedding.

In [ ]:
rh.sensitivity_table()

## 8. Cost

In [ ]:
rh.cost_table(t)

## 9. Choosing a version

Questions to answer from the tables above, for each version and rule:
1. Does it stay quiet before the change? (alarms before the change; CASE1 and CASE2)
2. Does it raise an alarm after the change, and how soon? (alarms after, delay)
3. Does it test the stream at all around the change? (share tested, runs that tested the change triple)
4. Do the models and compositions agree? (spread of the 10 runs)
5. What does it cost? (run time, re-embeds)
6. On the full data, do the peaks fall on dates that can be traced to papers?